# Лабораторная работа 5. Знакомство с современными LLM

#### Работу выполнил: {впишите вашу фамилию} {впишите ваше имя}, студент группы ФИб-4

# Чать 1. Знакомство с библиотекой `transformers`

Классы `AutoTokenizer` и `AutoModelForCausalLM` позволяет загрузить чекпоинты языковой модели и выполнить генерацию текста.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

В качестве модели для экспериментов возьмите модель `microsoft/Phi-3.5-mini-instruct` с huggingface. Веса загружайте в типе `torch.bfloat16` для экономии памяти GPU.

In [ ]:
model_name = "NousResearch/Hermes-3-Llama-3.2-3B"
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained(model_name)


Входной текст для генерации может быть представлен в режиме диалога в виде списка объектов с полями `role` и `content`. Значения поля `role` может принимать значения `system`, `user`, `assistent`, что соответсвует системному, пользовательскому промптам и ответу модели.

In [ ]:
chat = [
  {"role": "system", "content": "Отвечай на русском языке."},
  {"role": "user", "content": "Что ты думаешь о законах робототехники?"},
  {"role": "assistant", "content": "Законы робототехники — это нарастающая область правовой регуляции, которая стремится адаптироваться к быстрому развитию технологий в области искусственного интеллекта (ИИ) и автоматизации."},
  {"role": "user", "content": "А какой второй закон?"}
]
inputs = tokenizer.apply_chat_template(chat, tokenize=True, add_generation_prompt=True, return_tensors="pt")
print(tokenizer.decode(inputs[0]))

Генерация выполняется с помощью метода generate().

In [ ]:
output = model.generate(inputs.to(model.device), max_new_tokens=300)
tokenizer.decode(output[0])

Для вывода конечного сгенерировннного ответа потребуется постобработка.

Используя библиотеку `gradio` можно добавить графический интерфейс. Например, для чатбота будет полезен компонент `ChatInterface`.

**Задание 1.** Напишите чатбота с поддержкой истории при генерации ответа. Глубину истории нужно задавать в начале диалога в параметре `history_size`. Ведение диалога можно реализовать посредством графического компонента или через функцию `input()`.

In [ ]:
#ваш код

# Часть 2. Знакомство с `langchain`. Простой пример RAG

Установим требуемые библиотеки.

In [ ]:
!pip install langchain
!pip install -U langchain-community
!pip install jq
!pip install chromadb

Выполним импорт требуемых библиотек.

In [ ]:
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.document_loaders import DirectoryLoader, JSONLoader

Определим переменные для каталога с документами, по которым будет выполняться поиск, и для каталога под векторную базу данных.

In [ ]:
data_directory = "/content/documents"
db_directory = "/content/chroma_db"
load_new_db = False

Инициализируем модель, с помощью которой будут строиться эмбеддинги текстовых фагментов документов.

In [ ]:
data_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

Загрузим документы. Пусть документы представляют собой `json` файлы со списками объектов полем `text`, в котором хранится текст (пример файла находится в гугл-папке вместе с блокнотом лабораторной работы).

In [ ]:
loader = DirectoryLoader(data_directory, glob="**/*.json", show_progress=True, loader_cls=JSONLoader, loader_kwargs={"jq_schema":".[].text"})
documents = loader.load()
print(f"Count of documents is {len(documents)}")

Разделим тексты на фрагменты.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=10)
texts = text_splitter.split_documents(documents)
print(f"Count of hunks is {len(texts)}")

Наполним векторную базу данных.

In [ ]:
vector_db = Chroma.from_documents(documents=texts, embedding=data_embeddings, persist_directory=db_directory)

Создадим объект, через который будет выполняться поиск.

In [ ]:
retriever = vector_db.as_retriever(search_type="mmr")

Зададим запрос и выполним поиск.

In [ ]:
query = "Самая высокооплачиваемая писательница"
response = retriever.get_relevant_documents(query)
results = [{"id":i, "text":x.page_content, "source":x.metadata['source']} for i,x in enumerate(response)]
for x in results:
  print(x)

**Задание 2.** Реализуйте поиск по новостным текстам из датасета предыдущих лабораторных работ. Протестируйте работу поиска.

In [ ]:
# ва код

# Часть 3. Чатбот с использованием внешних знаний

**Задание 3.** Напишите чатбот, в котором перед генерацией выполнятся поиск внешних знаний, а затем выполняется генерация ответа пользователю с учётом этих знаний и истории диалога (тестируйте на глубине истории не более 5 сообщений диалога). Протестируйте работу чатбота.

In [ ]:
# ваш код